# 00 · Preparação: ambiente, dados públicos e mapa da apresentação

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


## Objetivos deste notebook
1. Verificar o ambiente (`uv`, pacote `replang`, JAX opcional).
2. Baixar/preparar os **dados públicos** usados em toda a sequência e entender de onde vêm.
3. Apresentar o **mapa da apresentação** (4h) e como os notebooks se encadeiam com os artigos.

## Mapa da apresentação (4 h)

| Bloco | Tempo | Pergunta-guia | Notebooks | Artigos |
|---|---|---|---|---|
| A. Premissas | 0:00–0:35 | Por que vetores? O que é "significado distribucional"? | 01 | Harris (1954); LSA/HAL; Levy & Goldberg (2014) |
| B. word2vec | 0:35–1:20 | Como um modelo log-linear aprende `rei − homem + mulher ≈ rainha`? | 02, 03 | Mikolov et al. (2013a, 2013b) |
| C. Variações | 1:20–1:50 | Contagem *vs.* predição (GloVe); morfologia (fastText); frases/documentos (Doc2Vec) | 04, 05, 11 | Pennington (2014); Bojanowski (2017); Le & Mikolov (2014) |
| *intervalo* | 1:50–2:05 | | | |
| D. Português | 2:05–2:45 | O que muda em PT? Como avaliar? Analogias servem? | 06, 07 | Hartmann et al. (2017) |
| E. Viés | 2:45–3:35 | O que mais o vetor carrega? Dá para tirar? | 08, 09, 10 | Bolukbasi et al. (2016) |
| F. Contexto e síntese | 3:35–4:00 | E se a palavra tiver um vetor por contexto? | 12, 13 | Peters et al. (2018) |

A interface Streamlit (`uv run replang app`) tem uma página por bloco para exploração ao vivo.

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook_connected" if os.environ.get("REPLANG_PLOTLY_CONNECTED") else "plotly_mimetype+notebook"
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: /home/user/linguagem-blis2s26 | fast: False


## 1. Ambiente

O projeto usa [`uv`](https://docs.astral.sh/uv/) para gerir o ambiente (`pyproject.toml` + `uv.lock`). Dependências principais:
`numpy`, `scipy`, `scikit-learn`, `gensim` (modelos "de produção"), `plotly` (figuras), `streamlit` (interface), `jax` (extra `contextual`, para o biLM).

In [2]:
import importlib
for pkg in ["numpy", "scipy", "sklearn", "gensim", "plotly", "streamlit", "jax"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<10} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"{pkg:<10} ausente ({type(e).__name__}) — opcional" if pkg == "jax" else f"{pkg:<10} ERRO: {e}")
import replang
print("replang", replang.__version__, "| dados em", PATHS.data)

numpy      2.4.6
scipy      1.17.1


sklearn    1.9.1
gensim     4.4.0
plotly     7.1.0


streamlit  1.64.0


jax        0.10.2
replang 0.1.0 | dados em /home/user/linguagem-blis2s26/data


## 2. Fontes de dados públicas

Todos os dados são públicos e têm licença permissiva ou são de domínio público. O módulo `replang.config.SOURCES` documenta cada um.

| Chave | Uso na apresentação | Artigo relacionado |
|---|---|---|
| `machado` | corpus PT-BR (2,4 M tokens) para treinar word2vec/GloVe/fastText/Doc2Vec/biLM **ao vivo** | todos |
| `mac_morpho` | POS tagging (avaliação extrínseca) | Hartmann et al. (2017) |
| `ptwiki2vec_100d` | embeddings PT pré-treinados na Wikipédia (Wikipedia2Vec, skip-gram 100d) | Hartmann; Bolukbasi (em PT) |
| `glove_en_50d` | embeddings EN para reproduzir Bolukbasi et al. | Bolukbasi et al. (2016) |
| `questions_words`, `lx_analogies_*` | analogias EN e PT-BR/PT-EU | Mikolov (2013a); Rodrigues (2016); Hartmann (2017) |
| `debiaswe` | pares definicionais, pares de equalização, profissões, palavras específicas de gênero | Bolukbasi et al. (2016) |
| `nilc` (conector opcional) | os 31 modelos de Hartmann et al. — exige download manual | Hartmann et al. (2017) |

> Os embeddings do NILC (`nilc.icmc.usp.br/embeddings`) não são acessíveis de todas as redes; por isso o padrão é o Wikipedia2Vec PT. O notebook 06 mostra como trocar.

In [3]:
from replang.config import SOURCES
pd.DataFrame([{"chave": k, "descrição": v["desc"], "licença": v["license"]} for k, v in SOURCES.items()])

,chave,descrição,licença
0,machado,Obra completa de Machado de Assis (domínio público) – corpus PT-BR literário,Domínio público (machado.mec.gov.br)
1,mac_morpho,Mac-Morpho: corpus PT-BR anotado com classes gramaticais (POS),CC BY 4.0 (NILC/USP)
2,floresta,Floresta Sintá(c)tica – treebank PT,Linguateca
3,ptwiki2vec_100d,"Wikipedia2Vec PT (skip-gram, 100d, Wikipédia 2018) – embeddings pré-treinados",Apache 2.0 (Wikipedia2Vec) / CC BY-SA (texto da Wikipédia)
4,glove_en_50d,GloVe 6B 50d (Wikipedia+Gigaword) empacotado pelo gensim-data,PDDL (Stanford GloVe)
5,questions_words,"Conjunto de analogias do word2vec (Mikolov et al., 2013a)",Apache 2.0
6,lx_analogies_br,"Analogias traduzidas para PT-BR (Rodrigues et al., 2016) – usadas por Hartma...","LX-DSemVectors (NLX, Univ. Lisboa)"
7,lx_analogies_eu,"Analogias traduzidas para PT-EU (Rodrigues et al., 2016)","LX-DSemVectors (NLX, Univ. Lisboa)"
8,debiaswe,"Listas de pares definicionais, pares de equalização, palavras específicas de...",MIT
9,nilc,"Repositório de embeddings PT do NILC (Hartmann et al., 2017): word2vec, GloV...",Uso acadêmico (NILC/USP)


### 2.1 Download e pré-processamento

`replang download` baixa os arquivos brutos para `data/raw`; `replang prepare` gera as versões processadas
(`data/processed`) e os caches truncados de embeddings (`data/samples`, versionados). Aqui chamamos as funções diretamente;
se os artefatos já existem, nada é baixado.

In [4]:
from replang.data.corpora import prepare_machado, prepare_macmorpho, load_machado_sentences, load_macmorpho
from replang.data.embeddings import load_ptwiki, load_glove_en

p_mach = prepare_machado(); p_mm = prepare_macmorpho()
sents = load_machado_sentences()
print(f"Machado de Assis: {len(sents):,} sentenças · {sum(map(len, sents)):,} tokens · exemplo: {' '.join(sents[1234][:14])} …")
mm = load_macmorpho()
print(f"Mac-Morpho: {len(mm):,} sentenças anotadas · exemplo: {mm[3][:8]}")
pt = load_ptwiki(); en = load_glove_en()
print(pt, "|", en)

Machado de Assis: 150,091 sentenças · 2,357,192 tokens · exemplo: talvez prefiras ouvir a minha tragédia intitulada o punhal de bruto …


Mac-Morpho: 50,925 sentenças anotadas · exemplo: [('A', 'ART'), ('degradação', 'N'), ('de', 'PREP'), ('as', 'ART'), ('terras', 'N'), ('por', 'PREP'), ('o', 'ART'), ('mau', 'ADJ')]


WordVectors(name='Wikipedia2Vec PT 100d', n=50000, dim=100) | WordVectors(name='GloVe 6B 50d (EN)', n=50000, dim=50)


### 2.2 Pré-processamento (seguindo Hartmann et al., 2017, §2.1)

As decisões de normalização afetam o vocabulário e, portanto, os vetores. Reproduzimos as do artigo:
minúsculas; numerais → `0`; URLs → `URL`; e-mails → `EMAIL`; clíticos hifenizados preservados (*machucou-se*);
sentenças com < 5 tokens descartadas; tipos raros (< 5 ocorrências) tratados como `UNKNOWN` (nos modelos, via `min_count`).

In [5]:
from replang.utils.text import normalize_text, tokenize, iter_sentences
exemplo = "Em 1899, Machado publicou Dom Casmurro. Veja em http://machado.mec.gov.br ou escreva para bentinho@capitu.br! Ele machucou-se."
print(normalize_text(exemplo))
print(tokenize(exemplo))
print(list(iter_sentences(exemplo, min_tokens=5)))

em 0000, machado publicou dom casmurro. veja em  url  ou escreva para  email ! ele machucou-se.
['em', '0000,', 'machado', 'publicou', 'dom', 'casmurro', 'veja', 'em', 'url', 'ou', 'escreva', 'para', 'email', 'ele', 'machucou-se']
[['em', '0000,', 'machado', 'publicou', 'dom', 'casmurro'], ['veja', 'em', 'url', 'ou', 'escreva', 'para', 'email']]


## 3. Primeira olhada: o que um embedding "sabe"

Antes de qualquer teoria, uma amostra do que vem pela frente. O método `most_similar` calcula o **cosseno** entre o vetor da
palavra e todos os outros; `analogy` resolve `a : b :: c : ?` por aritmética vetorial (Mikolov et al., 2013a).

In [6]:
for w in ["rei", "computador", "feliz", "capitu"]:
    print(f"{w:<12}", [x for x, _ in pt.most_similar(w, topn=6)] if w in pt else "fora do vocabulário")
print("homem : rei :: mulher : ?", pt.analogy("homem", "rei", "mulher", topn=3))
print("brasil : brasília :: frança : ?", pt.analogy("brasil", "brasília", "frança", topn=3))
print("man : king :: woman : ?", en.analogy("man", "king", "woman", topn=3))

rei          ['monarca', 'soberano', 'herdeiro', 'destronado', 'trono', 'rainha']
computador   ['computadores', 'hardware', 'portátil', 'dispositivo', 'microprocessador', 'emulador']
feliz        ['triste', 'contente', 'tranquilo', 'infeliz', 'felizes', 'querida']
capitu       fora do vocabulário
homem : rei :: mulher : ? [('rainha', 0.81949782371521), ('consorte', 0.7614757418632507), ('princesa', 0.7310948967933655)]
brasil : brasília :: frança : ? [('toulouse', 0.7129448652267456), ('avinhão', 0.6799794435501099), ('provence', 0.6797589063644409)]
man : king :: woman : ? [('queen', 0.8523508310317993), ('throne', 0.7664310336112976), ('prince', 0.7592698335647583)]


E uma amostra do **viés** que também vem junto (tema do bloco E):

In [7]:
from replang.data import EN, PT
print("EN  he : doctor :: she : ?", en.analogy("he", "doctor", "she", topn=3, exclude_words=EN.get().gender_specific))
print("PT  ele : médico :: ela : ?", pt.analogy("ele", "médico", "ela", topn=3, exclude_words=PT.get().gender_specific))
print("PT  homem : programador :: mulher : ?", pt.analogy("homem", "programador", "mulher", topn=3, exclude_words=PT.get().gender_specific))

EN  he : doctor :: she : ? [('nurse', 0.8434540629386902), ('child', 0.7920850515365601), ('pregnant', 0.772709310054779)]
PT  ele : médico :: ela : ? [('pediatra', 0.8375167846679688), ('psiquiatra', 0.7985932230949402), ('médica', 0.788758397102356)]
PT  homem : programador :: mulher : ? [('desenvolvedor', 0.6239131689071655), ('musicista', 0.594486653804779), ('diretora', 0.5922166109085083)]


## 4. Como os notebooks se organizam

Cada notebook segue o mesmo padrão: **premissas → mecanismo (com código *from scratch*) → experimento → o que o artigo encontrou → discussão**.
As implementações didáticas estão em `replang.models` (numpy); os modelos "de produção" (gensim) são treinados por `uv run replang train` e carregados de `data/models`.

## Referências
- MIKOLOV, T.; CHEN, K.; CORRADO, G.; DEAN, J. **Efficient Estimation of Word Representations in Vector Space.** ICLR Workshop, 2013. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- MIKOLOV, T.; SUTSKEVER, I.; CHEN, K.; CORRADO, G.; DEAN, J. **Distributed Representations of Words and Phrases and their Compositionality.** NIPS, 2013. [arXiv:1310.4546](https://arxiv.org/abs/1310.4546)
- PENNINGTON, J.; SOCHER, R.; MANNING, C. **GloVe: Global Vectors for Word Representation.** EMNLP, 2014. [ACL D14-1162](https://aclanthology.org/D14-1162/)
- BOJANOWSKI, P.; GRAVE, E.; JOULIN, A.; MIKOLOV, T. **Enriching Word Vectors with Subword Information.** TACL, 2017. [arXiv:1607.04606](https://arxiv.org/abs/1607.04606)
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)
- BOLUKBASI, T.; CHANG, K.-W.; ZOU, J.; SALIGRAMA, V.; KALAI, A. **Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.** NIPS, 2016. [arXiv:1607.06520](https://arxiv.org/abs/1607.06520)
- PETERS, M. et al. **Deep contextualized word representations.** NAACL, 2018. [arXiv:1802.05365](https://arxiv.org/abs/1802.05365)
- LE, Q.; MIKOLOV, T. **Distributed Representations of Sentences and Documents.** ICML, 2014. [arXiv:1405.4053](https://arxiv.org/abs/1405.4053)